# Save final netCDF for verification

In [2]:
import re
import os
import sys

import zarr
import yaml
from glob import glob
from datetime import datetime, timedelta

import numpy as np
import xarray as xr

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
sys.path.insert(0, os.path.realpath('../libs/'))
import verif_utils as vu

### Get the target data for coord reference

In [5]:
# fn_target = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404/C404_GP_2020.zarr'
# ds_target = xr.open_zarr(fn_target)

In [6]:
ds_geo = xr.open_zarr('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/static/C404_GP_static.zarr')
XLONG = ds_geo['XLONG'].values
XLAT = ds_geo['XLAT'].values

In [7]:
# flag_W = False

**CF-compliant (with W)**

In [8]:
# for exp_name in ['B3H', 'B6H', 'B12H']:
#     for year in range(2021, 2025):
#         # Load dataset
#         save_name1 = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/full_output/opt_{exp_name}_{year-1}_full.zarr'
#         ds_final1 = xr.open_zarr(save_name1, chunks={}).sel(time=slice(f'{year-1}-10-01T00', f'{year-1}-12-31T23'))
        
#         save_name2 = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/full_output/opt_{exp_name}_{year}_full.zarr'
#         ds_final2 = xr.open_zarr(save_name2, chunks={}).sel(time=slice(f'{year}-01-01T00', f'{year}-09-30T23'))
        
#         ds_final = xr.concat([ds_final1, ds_final2], dim='time')
        
#         # ========================================= #
#         # non-negative variable fix
#         vars_to_fix = ['WRF_PWAT_05', 'WRF_Q_tot_05', 'WRF_precip_025', 'WRF_radar_composite_025', 'WRF_GLW', 'WRF_SWDOWN']
        
#         for var in vars_to_fix:
#             ds_final[var] = ds_final[var].clip(min=0)
        
#         # [0, 1] variable fix
#         vars_to_fix = ['WRF_SMOIS', 'WRF_TCC']
#         for var in vars_to_fix:
#             ds_final[var] = ds_final[var].clip(min=0).clip(max=1)
        
#         # ========================================= #
#         # solar radiation fix
#         ds_solar1 = xr.open_zarr(f'/glade/campaign/ral/hap/ksha/DWC_data/CONUS_domain_GP/solar/solar_GP_{year-1}.zarr')
#         ds_solar2 = xr.open_zarr(f'/glade/campaign/ral/hap/ksha/DWC_data/CONUS_domain_GP/solar/solar_GP_{year}.zarr')
        
#         ds_solar = xr.concat([ds_solar1, ds_solar2], dim='time')
#         ds_solar_ref = ds_solar.sel(time=ds_final['time'])
        
#         eps = 1e-6
#         is_night = (ds_solar_ref['TSI'].fillna(0) <= eps).all(dim=('south_north', 'west_east'))
#         ds_final['WRF_SWDOWN'] = ds_final['WRF_SWDOWN'].where(~is_night, other=0)
        
#         # ========================================= #
#         # square variables and drop originals
#         ds_final = ds_final.assign(
#             WRF_PWAT=ds_final['WRF_PWAT_05'] ** 2,
#             WRF_Q_tot=ds_final['WRF_Q_tot_05'] ** 2
#         ).drop_vars(['WRF_PWAT_05', 'WRF_Q_tot_05'])
        
#         ds_final['WRF_precip'] = ds_final['WRF_precip_025']**4
#         ds_final['WRF_radar_composite'] = ds_final['WRF_radar_composite_025']**4
#         ds_final = ds_final.drop_vars(['WRF_precip_025', 'WRF_radar_composite_025'])
        
#         # ========================================= #
#         # Rename dimensions
#         ds_final = ds_final.rename({
#             'bottom_top': 'level',
#             'south_north': 'latitude',
#             'west_east': 'longitude'
#         })
        
#         # Subset time
#         #ds_final = ds_final.isel(time=slice(None, -1))
        
#         # Assign level coordinate
#         ds_final = ds_final.assign_coords(level=[0, 3, 6, 9, 12, 15, 18, 21, 24, 30, 36, 42])
        
#         # Drop old lat/lon and assign new ones
#         ds_final = ds_final.drop_vars(['latitude', 'longitude'], errors='ignore').assign_coords(
#             XLAT=(('latitude', 'longitude'), XLAT),
#             XLONG=(('latitude', 'longitude'), XLONG)
#         )

#         if flag_W:
#             # ========================================================= #
#             # WRF_W
            
#             # Build a DataArray aligned to the dataset's 'level' coordinate
#             R_da = xr.DataArray(
#                 R_adjust,
#                 dims=["level"],
#                 coords={"level": ds_final["level"].values},
#                 name="R_adjust"
#             )
            
#             ds = ds_final
            
#             lat2d = ds['XLAT']              # (latitude, longitude)
#             lon2d = ds['XLONG']             # (latitude, longitude)
#             latr  = np.deg2rad(lat2d)
#             lonr  = np.deg2rad(lon2d)
            
#             # Spacing along longitude (x-direction): shape (latitude, longitude-1)
#             dlon = lonr.diff('longitude')
#             lat_mid_lon = 0.5 * (latr.isel(longitude=slice(1, None)) + latr.isel(longitude=slice(None, -1)))
#             dx = R_earth * np.cos(lat_mid_lon) * dlon  # meters
            
#             # Spacing along latitude (y-direction): shape (latitude-1, longitude)
#             dlat = latr.diff('latitude')
#             dy = R_earth * dlat  # meters
            
#             du_dx = _central_derivative(ds['WRF_U'], dx, 'longitude')   # s^-1
#             dv_dy = _central_derivative(ds['WRF_V'], dy, 'latitude')    # s^-1
#             div_h = du_dx + dv_dy    
            
#             # p = ds['WRF_P']  # Pa
            
#             omega = xr.apply_ufunc(
#                 _omega_from_divergence, div_h, ds['WRF_P'],
#                 input_core_dims=[['level'], ['level']],
#                 output_core_dims=[['level']],
#                 vectorize=True,
#                 dask='parallelized',
#                 output_dtypes=[ds['WRF_P'].dtype]
#             )
            
#             ds = ds.assign(WRF_OMEGA=omega)
#             T = ds['WRF_T']  # Kelvin
#             qv = ds['WRF_Q_tot']
            
#             Tv = T * (1.0 + 0.61 * qv) if qv is not None else T
#             rho = ds['WRF_P'] / (Rd * Tv)                 # kg m^-3
#             w = - ds['WRF_OMEGA'] / (rho * g)             # m s^-1 (positive up)
            
#             ds = ds.assign(WRF_W=w)
#             ds = ds.assign(WRF_W = ds["WRF_W"] * R_da)
#             ds['WRF_W'] = ds['WRF_W'].transpose('time', 'level', 'latitude', 'longitude')
            
#             ds_final = ds.drop_vars(['WRF_OMEGA'])
            
#             # ========================================================= #
        
#         # Add global attribute
#         ds_final.attrs['Conventions'] = 'CF-1.11'
        
#         # Decode time
#         ds_final['time'] = xr.decode_cf(ds_final[['time']]).time
        
#         # Replace indexing dims with coordinate dims and standard names
#         if 'XLAT' in ds_final and 'XLONG' in ds_final:
#             ds_final = ds_final.set_coords(['XLAT', 'XLONG'])
#             ds_final['XLAT'] = ds_final['XLAT'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
#             ds_final['XLONG'] = ds_final['XLONG'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
        
#             ds_final['XLAT'].attrs.update({
#                 "standard_name": "latitude",
#                 "units": "degrees_north",
#                 "long_name": "latitude"
#             })
#             ds_final['XLONG'].attrs.update({
#                 "standard_name": "longitude",
#                 "units": "degrees_east",
#                 "long_name": "longitude"
#             })
        
#         # Replace lat/lon dims with south_north/west_east
#         dim_map_3d = ('time', 'level', 'south_north', 'west_east')
#         dim_map_2d = ('time', 'south_north', 'west_east')
        
#         for var in ds_final.data_vars:
#             dims = ds_final[var].dims
#             dim_rename = {}
#             if 'latitude' in dims:
#                 dim_rename['latitude'] = 'south_north'
#             if 'longitude' in dims:
#                 dim_rename['longitude'] = 'west_east'
            
#             if dim_rename:
#                 ds_final[var] = ds_final[var].rename(dim_rename)
        
#         output_name = f'/glade/derecho/scratch/ksha/GWC_Results/final_{exp_name}_{year}_WY.nc'
#         ds_final.to_netcdf(output_name, format='NETCDF4_CLASSIC')
        
#         print(output_name)
#         raise

**CF-compliant from zarr**

In [9]:
# for exp_name in ['B3H', 'B6H', 'GDAS']:
#     for year in range(2021, 2025):
#         ds_final = xr.open_zarr(f'/glade/campaign/ral/hap/ksha/GWC_results/FINAL_run/final_{exp_name}_{year}_WY.zarr', chunks={})
        
#         # Add global attribute
#         ds_final.attrs['Conventions'] = 'CF-1.11'
        
#         # Decode time
#         ds_final['time'] = xr.decode_cf(ds_final[['time']]).time

#         # ========================================= #
#         # Rename dimensions
#         ds_final = ds_final.rename({
#             'south_north': 'latitude',
#             'west_east': 'longitude'
#         })
        
#         # Subset time
#         # ds_final = ds_final.isel(time=slice(None, -1))
        
#         # # Assign level coordinate
#         # ds_final = ds_final.assign_coords(level=[0, 3, 6, 9, 12, 15, 18, 21, 24, 30, 36, 42])
        
#         # Drop old lat/lon and assign new ones
#         ds_final = ds_final.drop_vars(['latitude', 'longitude'], errors='ignore').assign_coords(
#             XLAT=(('latitude', 'longitude'), XLAT),
#             XLONG=(('latitude', 'longitude'), XLONG)
#         )

        
#         # Replace indexing dims with coordinate dims and standard names
#         if 'XLAT' in ds_final and 'XLONG' in ds_final:
#             ds_final = ds_final.set_coords(['XLAT', 'XLONG'])
#             ds_final['XLAT'] = ds_final['XLAT'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
#             ds_final['XLONG'] = ds_final['XLONG'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
        
#             ds_final['XLAT'].attrs.update({
#                 "standard_name": "latitude",
#                 "units": "degrees_north",
#                 "long_name": "latitude"
#             })
#             ds_final['XLONG'].attrs.update({
#                 "standard_name": "longitude",
#                 "units": "degrees_east",
#                 "long_name": "longitude"
#             })
        
#         # Replace lat/lon dims with south_north/west_east
#         dim_map_3d = ('time', 'level', 'south_north', 'west_east')
#         dim_map_2d = ('time', 'south_north', 'west_east')
        
#         for var in ds_final.data_vars:
#             dims = ds_final[var].dims
#             dim_rename = {}
#             if 'latitude' in dims:
#                 dim_rename['latitude'] = 'south_north'
#             if 'longitude' in dims:
#                 dim_rename['longitude'] = 'west_east'
            
#             if dim_rename:
#                 ds_final[var] = ds_final[var].rename(dim_rename)
        
#         output_name = f'/glade/derecho/scratch/ksha/GWC_Results/final_{exp_name}_{year}_WY.nc'
#         ds_final.to_netcdf(output_name, format='NETCDF4_CLASSIC')
        
#         print(output_name)

### Daily fields

In [11]:
for exp_name in ['B3H', 'B6H', 'GDAS',]:
    for i_year, year in enumerate(range(2021, 2025)):
        fn = f'/glade/campaign/ral/hap/ksha/GWC_results/FINAL_run/final_{exp_name}_{year}_WY_daily.zarr'
        ds_final = xr.open_zarr(fn)
        
        # Add global attribute
        ds_final.attrs['Conventions'] = 'CF-1.11'
        
        # Decode time
        ds_final['time'] = xr.decode_cf(ds_final[['time']]).time
        
        # ========================================= #
        # Rename dimensions
        ds_final = ds_final.rename({
            'south_north': 'latitude',
            'west_east': 'longitude'
        })
        
        # Subset time
        # ds_final = ds_final.isel(time=slice(None, -1))
        
        # # Assign level coordinate
        # ds_final = ds_final.assign_coords(level=[0, 3, 6, 9, 12, 15, 18, 21, 24, 30, 36, 42])
        
        # Drop old lat/lon and assign new ones
        ds_final = ds_final.drop_vars(['latitude', 'longitude'], errors='ignore').assign_coords(
            XLAT=(('latitude', 'longitude'), XLAT),
            XLONG=(('latitude', 'longitude'), XLONG)
        )
        
        # Replace indexing dims with coordinate dims and standard names
        if 'XLAT' in ds_final and 'XLONG' in ds_final:
            ds_final = ds_final.set_coords(['XLAT', 'XLONG'])
            ds_final['XLAT'] = ds_final['XLAT'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
            ds_final['XLONG'] = ds_final['XLONG'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
        
            ds_final['XLAT'].attrs.update({
                "standard_name": "latitude",
                "units": "degrees_north",
                "long_name": "latitude"
            })
            ds_final['XLONG'].attrs.update({
                "standard_name": "longitude",
                "units": "degrees_east",
                "long_name": "longitude"
            })
        
        # Replace lat/lon dims with south_north/west_east
        dim_map_3d = ('time', 'level', 'south_north', 'west_east')
        dim_map_2d = ('time', 'south_north', 'west_east')
        
        for var in ds_final.data_vars:
            dims = ds_final[var].dims
            dim_rename = {}
            if 'latitude' in dims:
                dim_rename['latitude'] = 'south_north'
            if 'longitude' in dims:
                dim_rename['longitude'] = 'west_east'
            
            if dim_rename:
                ds_final[var] = ds_final[var].rename(dim_rename)
                
        output_name = f'/glade/derecho/scratch/ksha/GWC_Results/final_{exp_name}_{year}_WY_daily.nc'
        ds_final.to_netcdf(output_name, format='NETCDF4_CLASSIC')
        
        print(output_name)

/glade/derecho/scratch/ksha/GWC_Results/final_B3H_2021_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/final_B3H_2022_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/final_B3H_2023_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/final_B3H_2024_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/final_B6H_2021_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/final_B6H_2022_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/final_B6H_2023_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/final_B6H_2024_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/final_GDAS_2021_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/final_GDAS_2022_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/final_GDAS_2023_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/final_GDAS_2024_WY_daily.nc


In [12]:
ds_final

<xarray.Dataset>
Dimensions:     (time: 365, south_north: 336, west_east: 336)
Coordinates:
  * time        (time) datetime64[ns] 2023-10-02 2023-10-03 ... 2024-09-30
    XLAT        (south_north, west_east) float32 27.87 27.87 ... 39.64 39.64
    XLONG       (south_north, west_east) float32 -102.5 -102.4 ... -87.39 -87.34
Dimensions without coordinates: south_north, west_east
Data variables:
    WRF_TMAX    (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_TMIN    (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_precip  (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.11

### To netCDF target

In [1]:
# for year in range(2021, 2025):
#     # Load dataset lazily
#     ds_C404 = xr.open_zarr(
#         f'/glade/campaign/ral/hap/ksha/GWC_results/FINAL_run/target_{year}_WY.zarr', chunks={}
#     )

#     ds_C404 = ds_C404.drop_vars(['level'])
    
#     # Rename dimensions
#     ds_C404 = ds_C404.rename({
#         'bottom_top': 'level',
#         'south_north': 'latitude',
#         'west_east': 'longitude'
#     })
    
#     # Assign level coordinate
#     ds_C404 = ds_C404.assign_coords(level=[0, 3, 6, 9, 12, 15, 18, 21, 24, 30, 36, 42])
    
#     # Drop old lat/lon coords if present, then assign new ones
#     ds_C404 = ds_C404.drop_vars(['latitude', 'longitude'], errors='ignore').assign_coords(
#         XLAT=(('latitude', 'longitude'), XLAT),
#         XLONG=(('latitude', 'longitude'), XLONG)
#     )
    
#     # Set global attribute
#     ds_C404.attrs['Conventions'] = 'CF-1.11'
    
#     # Decode CF time
#     ds_C404['time'] = xr.decode_cf(ds_C404[['time']]).time
    
#     # Set coordinate variables and rename dims
#     if 'XLAT' in ds_C404 and 'XLONG' in ds_C404:
#         ds_C404 = ds_C404.set_coords(['XLAT', 'XLONG'])
    
#         # Rename only the coordinate dims, not data
#         ds_C404['XLAT'] = ds_C404['XLAT'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
#         ds_C404['XLONG'] = ds_C404['XLONG'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
    
#         # Set CF metadata
#         ds_C404['XLAT'].attrs.update({
#             "standard_name": "latitude",
#             "units": "degrees_north",
#             "long_name": "latitude"
#         })
#         ds_C404['XLONG'].attrs.update({
#             "standard_name": "longitude",
#             "units": "degrees_east",
#             "long_name": "longitude"
#         })
    
#     # Rename lat/lon dims in data variables
#     for var in ds_C404.data_vars:
#         dims = ds_C404[var].dims
#         rename_dims = {}
#         if 'latitude' in dims:
#             rename_dims['latitude'] = 'south_north'
#         if 'longitude' in dims:
#             rename_dims['longitude'] = 'west_east'
#         if rename_dims:
#             ds_C404[var] = ds_C404[var].rename(rename_dims)
            
#     output_name = f'/glade/derecho/scratch/ksha/GWC_Results/target_{year}_WY.nc'
#     ds_C404.to_netcdf(output_name, format='NETCDF4_CLASSIC')
#     print(output_name)

In [15]:
for year in range(2021, 2025):
    # Load dataset lazily
    ds_C404 = xr.open_zarr(
        f'/glade/campaign/ral/hap/ksha/GWC_results/FINAL_run/target_{year}_WY_daily.zarr', chunks={}
    )
    
    # Rename dimensions
    ds_C404 = ds_C404.rename({
        'south_north': 'latitude',
        'west_east': 'longitude'
    })
    
    # Drop old lat/lon coords if present, then assign new ones
    ds_C404 = ds_C404.drop_vars(['latitude', 'longitude'], errors='ignore').assign_coords(
        XLAT=(('latitude', 'longitude'), XLAT),
        XLONG=(('latitude', 'longitude'), XLONG)
    )
    
    # Set global attribute
    ds_C404.attrs['Conventions'] = 'CF-1.11'
    
    # Decode CF time
    ds_C404['time'] = xr.decode_cf(ds_C404[['time']]).time
    
    # Set coordinate variables and rename dims
    if 'XLAT' in ds_C404 and 'XLONG' in ds_C404:
        ds_C404 = ds_C404.set_coords(['XLAT', 'XLONG'])
    
        # Rename only the coordinate dims, not data
        ds_C404['XLAT'] = ds_C404['XLAT'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
        ds_C404['XLONG'] = ds_C404['XLONG'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
    
        # Set CF metadata
        ds_C404['XLAT'].attrs.update({
            "standard_name": "latitude",
            "units": "degrees_north",
            "long_name": "latitude"
        })
        ds_C404['XLONG'].attrs.update({
            "standard_name": "longitude",
            "units": "degrees_east",
            "long_name": "longitude"
        })
    
    # Rename lat/lon dims in data variables
    for var in ds_C404.data_vars:
        dims = ds_C404[var].dims
        rename_dims = {}
        if 'latitude' in dims:
            rename_dims['latitude'] = 'south_north'
        if 'longitude' in dims:
            rename_dims['longitude'] = 'west_east'
        if rename_dims:
            ds_C404[var] = ds_C404[var].rename(rename_dims)
            
    output_name = f'/glade/derecho/scratch/ksha/GWC_Results/target_{year}_WY_daily.nc'
    ds_C404.to_netcdf(output_name, format='NETCDF4_CLASSIC')
    print(output_name)

/glade/derecho/scratch/ksha/GWC_Results/target_2021_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/target_2022_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/target_2023_WY_daily.nc
/glade/derecho/scratch/ksha/GWC_Results/target_2024_WY_daily.nc
